# A.2: Training on tiny Shakespeare

Train a 512-entry tokenizer on 90 percent of the corpus, show the first and last merges, and encode a line.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 The complete tokenizer (`bpe.py`)
import regex as re

from collections import Counter

# GPT-2's pre-tokenization pattern (Radford et al. 2019).
GPT2_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def merge(ids, pair, new_id):
    """Replace every occurrence of `pair` in the tuple `ids` with `new_id`."""
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return tuple(out)

class ByteBPE:
    def __init__(self, pattern=GPT2_PATTERN):
        self.pattern = re.compile(pattern)
        self.merges = {}                               # (id, id) -> new id
        self.vocab = {i: bytes([i]) for i in range(256)}  # id -> bytes
        self.special = {}                              # str -> id
        self.byte_ids = list(range(256))               # byte value -> id

    # ---- training: Steps 1 and 2 ---------------------------------------
    def train(self, text, vocab_size):
        chunks = Counter(self.pattern.findall(text))
        words = {tuple(c.encode("utf-8")): n for c, n in chunks.items()}
        for new_id in range(256, vocab_size):
            stats = Counter()
            for w, n in words.items():
                for pair in zip(w, w[1:]):
                    stats[pair] += n
            if not stats:
                break
            pair = max(stats, key=stats.get)   # ties: first pair seen
            words = {merge(w, pair, new_id): n for w, n in words.items()}
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]

    # ---- encoding: Step 3 ----------------------------------------------
    def _encode_chunk(self, ids):
        while len(ids) >= 2:
            # Merge the pair that was learned earliest. Merge IDs grow with
            # rank, so the pair with the smallest merge ID has the lowest rank.
            pair = min(zip(ids, ids[1:]),
                       key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return list(ids)

    def encode_ordinary(self, text):
        ids = []
        for chunk in self.pattern.findall(text):
            ids.extend(self._encode_chunk(
                tuple(self.byte_ids[b] for b in chunk.encode("utf-8"))))
        return ids

    def encode(self, text, allow_special=False):
        if not (allow_special and self.special):
            return self.encode_ordinary(text)
        split = "(" + "|".join(re.escape(s) for s in self.special) + ")"
        ids = []
        for part in re.split(split, text):
            if part in self.special:
                ids.append(self.special[part])
            elif part:
                ids.extend(self.encode_ordinary(part))
        return ids

    # ---- special tokens and decoding: Steps 4 and 5 --------------------
    def add_special_tokens(self, tokens):
        for t in tokens:
            if t not in self.special:
                new_id = len(self.vocab)
                self.special[t] = new_id
                self.vocab[new_id] = t.encode("utf-8")

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8", errors="replace")


In [ ]:
import os, urllib.request

if not os.path.exists("shakespeare.txt"):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "shakespeare.txt")
text = open("shakespeare.txt", encoding="utf-8").read()
split = int(0.9 * len(text))
train, held_out = text[:split], text[split:]

tok = ByteBPE()
tok.train(train, vocab_size=512)
print([tok.vocab[i] for i in range(256, 276)])
print([tok.vocab[i] for i in range(500, 512)])


**Expected output**

Output:

```text
[b' t', b'he', b' a', b'ou', b' s', b' m', b'in', b' w', b're', b'ha', b' the', b'nd', b' b', b'is', b'or', b' f', b'er', b'll', b'it', b'on']
[b'ond', b' their', b'self', b' there', b' know', b' king', b' if', b'ep', b'ind', b' tr', b' br', b' O']
```


In [ ]:
line = "To be, or not to be: that is the question."
ids = tok.encode(line)
print(len(ids), ids)
print([tok.vocab[i].decode("utf-8") for i in ids])


**Expected output**

Output:

```text
18 [396, 304, 44, 32, 270, 321, 287, 304, 58, 322, 326, 266, 32, 113, 117, 377, 395, 46]
['To', ' be', ',', ' ', 'or', ' not', ' to', ' be', ':', ' that', ' is', ' the', ' ', 'q', 'u', 'est', 'ion', '.']
```
